# LoRA fine-tune of a small open model for FPGA-synthesis regression triage

Fine-tunes **Qwen/Qwen2.5-1.5B-Instruct** (Apache-2.0, not gated) with LoRA to classify regression failure clusters from the
`agentic-fpga-tool-verification` PoC as `tool_defect`, `test_bug`, `infrastructure` or `known_issue`, then evaluates
the base model and the fine-tuned model on the PoC's 22-item gold set with the same prompt, parsing and metrics as the
PoC (`vagent/llm.py`, `vagent/evaluate.py`).

* Training data: 309 train + 71 validation examples generated by `finetune/make_dataset.py` from the PoC's own
  fault injection (Yosys jobs with injected tool defects, testcase bugs and emulated infrastructure faults); labels come
  from the injected ground truth, never from the rule classifier. Gold-set items and their near-duplicates were removed.
* Test set: the 22 gold items, reported separately for the 10 harvested items (same fault families as training), the
  12 handwritten hard cases, and the 6 handwritten items whose decision pattern has no look-alike in the training
  data (`hw, no twin`). The handwritten columns are the honest headline; one item is 4.5 points of 22.
* Everything the notebook needs is embedded below; it downloads only Python packages and the base model.

**Run**: Kaggle: Settings > Accelerator > GPU T4 x2, Internet on, then Run all. Colab: Runtime > Change runtime type >
T4 GPU, then Runtime > Run all. Expected time on one T4: roughly 15 to 30 minutes (an estimate from the token
counts, not a measurement). Outputs: `results.json`, the LoRA adapter (zip) and a merged fp16 model; an optional last
cell converts the merged model to GGUF for llama.cpp.


In [ ]:
import os, sys, json, math, time, platform, subprocess

SMOKE = os.environ.get("TRIAGE_SMOKE") == "1"  # tiny CPU run used to test this notebook before release
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")  # Kaggle "GPU T4 x2": use one GPU, no DataParallel
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

MODEL_ID, MODEL_REVISION = 'Qwen/Qwen2.5-1.5B-Instruct', '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
SEED = 42
EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE, GRAD_ACCUM = 4, 4
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
MAX_LEN = 1536
MAX_NEW_TOKENS = 256
GEN_BATCH = 8
VAL_EVAL_MAX = 80
MAX_STEPS = -1
TRAIN_LIMIT = TEST_LIMIT = None
RUN_GGUF = False
LLAMA_CPP_TAG = 'b12651'
if SMOKE:
    MODEL_ID, MODEL_REVISION = 'Qwen/Qwen2.5-0.5B-Instruct', '7ae557604adf67be50417f59c2c2f167def9a775'
    MAX_STEPS, BATCH_SIZE, GRAD_ACCUM, GEN_BATCH = 3, 1, 2, 2
    TRAIN_LIMIT, TEST_LIMIT, VAL_EVAL_MAX = 12, 4, 2
    RUN_GGUF = os.environ.get("TRIAGE_SMOKE_GGUF") == "1"
OUT_DIR = os.environ.get("TRIAGE_OUT") or (
    "/kaggle/working/triage_lora" if os.path.isdir("/kaggle/working") else os.path.abspath("triage_lora"))
os.makedirs(OUT_DIR, exist_ok=True)
TIMES = {}
T_START = time.time()
print("model", MODEL_ID, "| smoke" if SMOKE else "", "| outputs in", OUT_DIR)


## 1. Install pinned libraries (uses the preinstalled CUDA build of torch)

In [ ]:
import importlib.metadata as md

PINS = {'transformers': '5.22.2', 'peft': '0.21.3', 'trl': '1.17.1', 'datasets': '5.1.0', 'accelerate': '1.15.0'}


def installed(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None


need = [f"{p}=={v}" for p, v in PINS.items() if installed(p) != v]
if need:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *need])
print({p: installed(p) for p in PINS})
assert all(installed(p) == v for p, v in PINS.items()), "pinned versions are not active: restart the session, run all"


## 2. Embedded data (train, validation, test) and the PoC code used for parsing and metrics

In [ ]:
# train/val/test generated by finetune/make_dataset.py, embedded by finetune/build_notebook.py
import base64, gzip, hashlib
from collections import Counter

DATA_B64 = '...'
_raw = gzip.decompress(base64.b64decode(DATA_B64))
DATA_SHA256 = hashlib.sha256(_raw).hexdigest()
DATA = json.loads(_raw)
train_rows, val_rows, test_rows, DATASET_REPORT = DATA["train"], DATA["val"], DATA["test"], DATA["report"]
SYSTEM = DATASET_REPORT["system_prompt"]
print("data sha256", DATA_SHA256)
for name, rows in (("train", train_rows), ("val", val_rows), ("test", test_rows)):
    print(f"{name:5} {len(rows):4}", dict(sorted(Counter(r["label"] for r in rows).items())))
print("test by source", dict(Counter(r["source"] for r in test_rows)))
print("leakage checks", json.dumps(DATASET_REPORT["leakage"], indent=1))


In [ ]:
# Copied verbatim from the PoC (vagent/models.py, vagent/llm.py, vagent/evaluate.py) so parsing and metrics match.
import re
from typing import Literal, TypeVar
from pydantic import BaseModel, Field, ValidationError

Category = Literal["tool_defect", "test_bug", "infrastructure", "known_issue"]
CATEGORIES = ('tool_defect', 'test_bug', 'infrastructure', 'known_issue')
T = TypeVar("T", bound=BaseModel)

class TriageDecision(BaseModel):
    category: Category
    confidence: float = Field(ge=0.0, le=1.0)
    component: str = Field(min_length=1, max_length=120)
    rationale: str = Field(min_length=1, max_length=1200)
    needs_human_review: bool = False

def parse_structured(text: str, schema: type[T]) -> T:
    t = text.strip()
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t)
    if not t.startswith("{"):
        m = re.search(r"\{.*\}", t, re.S)
        t = m.group(0) if m else t
    return schema.model_validate_json(t)

def metrics(pairs: list[tuple[str, str]]) -> dict:
    """pairs of (label, predicted) -> accuracy, per-class precision/recall/F1, confusion matrix."""
    n = len(pairs)
    acc = sum(l == p for l, p in pairs) / n if n else 0.0
    per = {}
    for c in CATEGORIES:
        tp = sum(l == c and p == c for l, p in pairs)
        fp = sum(l != c and p == c for l, p in pairs)
        fn = sum(l == c and p != c for l, p in pairs)
        prec = tp / (tp + fp) if tp + fp else None
        rec = tp / (tp + fn) if tp + fn else None
        f1 = 2 * prec * rec / (prec + rec) if prec and rec else (0.0 if prec is not None and rec is not None else None)
        per[c] = {"support": tp + fn, "precision": _r(prec), "recall": _r(rec), "f1": _r(f1)}
    conf = {l: {p: sum(1 for a, b in pairs if a == l and b == p) for p in CATEGORIES} for l in CATEGORIES}
    return {"n": n, "accuracy": round(acc, 3), "per_class": per, "confusion": conf}

def _r(x):
    return None if x is None else round(x, 3)

SCHEMA_JSON = json.dumps(TriageDecision.model_json_schema())
assert all(SCHEMA_JSON in r["prompt_triage"] and SCHEMA_JSON in r["prompt_eval"] for r in test_rows)
assert all(SCHEMA_JSON in r["prompt"] for r in train_rows + val_rows)
print("prompt schema matches TriageDecision")


## 3. Base model, prompt format and token lengths

Prompts are the exact strings the PoC sends through `llm.structured` (task + TriageDecision JSON schema + evidence JSON), with the PoC's system prompt, wrapped in the model's chat template.

In [ ]:
import random
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda" and not SMOKE:
    raise RuntimeError("No GPU. Kaggle: Settings > Accelerator > GPU T4 x2. Colab: Runtime > Change runtime type > T4 GPU.")
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32  # T4 has no bf16
ENV = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
       "device": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
       "host": "kaggle" if os.path.isdir("/kaggle") else ("colab" if os.path.isdir("/content") else platform.platform()),
       **{p: installed(p) for p in PINS}}
print(ENV)

t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REVISION, dtype=DTYPE).to(DEVICE)
TIMES["load_model_s"] = round(time.time() - t0, 1)


def greedy(m):
    # the model ships sampling defaults and repetition_penalty 1.1; the PoC calls its LLM at temperature 0
    g = m.generation_config
    g.do_sample, g.temperature, g.top_p, g.top_k, g.repetition_penalty = False, None, None, None, 1.0
    g.max_new_tokens, g.pad_token_id = MAX_NEW_TOKENS, tok.pad_token_id
    return m


print("generation:", greedy(model).generation_config.to_diff_dict())


def messages(prompt):
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}]


def chat_text(prompt):
    return tok.apply_chat_template(messages(prompt), tokenize=False, add_generation_prompt=True)


def full_len(r):
    text = tok.apply_chat_template(messages(r["prompt"]) + [{"role": "assistant", "content": r["target"]}],
                                   tokenize=False)
    return len(tok(text, add_special_tokens=False).input_ids)


lengths = [full_len(r) for r in train_rows + val_rows]
test_lengths = [len(tok(chat_text(r[k]), add_special_tokens=False).input_ids) for r in test_rows
                for k in ("prompt_triage", "prompt_eval")]
TOKENS = {"train_val_max": max(lengths), "train_val_p95": int(np.percentile(lengths, 95)),
          "train_val_mean": round(float(np.mean(lengths)), 1), "test_prompt_max": max(test_lengths)}
print("tokens per example", TOKENS)
assert TOKENS["train_val_max"] <= MAX_LEN, "an example is longer than MAX_LEN and would be truncated"


In [ ]:
@torch.no_grad()
def generate(m, prompts):
    m.eval()
    tok.padding_side = "left"
    outs = []
    for i in range(0, len(prompts), GEN_BATCH):
        enc = tok([chat_text(p) for p in prompts[i:i + GEN_BATCH]], return_tensors="pt", padding=True,
                  add_special_tokens=False).to(DEVICE)
        gen = greedy(m).generate(**enc)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs


def predict(m, rows, key):
    raws = generate(m, [r[key] for r in rows])
    preds = []
    for raw in raws:
        try:
            preds.append(parse_structured(raw, TriageDecision).category)
        except (ValidationError, ValueError):
            preds.append("invalid")
    return preds, raws


TWINS = DATASET_REPORT["gold_twins_in_train"]
SUBSETS = ("all", "harvested", "handwritten", "handwritten_no_pattern_twin")


def in_subset(r, s):
    if s == "all":
        return True
    if s == "handwritten_no_pattern_twin":  # no training example has the same decision-relevant shape
        return r.get("source") == "handwritten" and TWINS[r["id"]]["pattern_twins"] == 0
    return r.get("source") == s


def summarize(rows, preds, subsets=SUBSETS):
    out = {}
    for s in subsets:
        idx = [i for i, r in enumerate(rows) if in_subset(r, s)]
        if not idx:
            continue
        m = metrics([(rows[i]["label"], preds[i]) for i in idx])
        m["correct"] = sum(rows[i]["label"] == preds[i] for i in idx)
        m["invalid"] = sum(preds[i] == "invalid" for i in idx)
        m["invalid_rate"] = round(m["invalid"] / len(idx), 3)
        # the PoC's gold evaluation falls back to the rules when the LLM output does not validate
        m["accuracy_with_rules_fallback"] = metrics(
            [(rows[i]["label"], rows[i]["rules_pred"] if preds[i] == "invalid" else preds[i]) for i in idx])["accuracy"]
        out[s] = m
    return out


if TEST_LIMIT:
    half = TEST_LIMIT // 2
    TEST = [r for r in test_rows if r["source"] == "harvested"][:half] + \
           [r for r in test_rows if r["source"] == "handwritten"][:TEST_LIMIT - half]
else:
    TEST = test_rows
VAL_EVAL = val_rows[:VAL_EVAL_MAX]
PRED, RAW, RESULTS = {}, {}, {"test": {}, "val": {}}
PRED["rules (vagent.triage.classify_rules)"] = [r["rules_pred"] for r in TEST]
for ref in DATASET_REPORT["reference_runs"]:
    PRED[ref + ", eval prompt"] = [r["ref_preds"].get(ref) or "invalid" for r in TEST]
for name, preds in PRED.items():
    RESULTS["test"][name] = summarize(TEST, preds)


def table():
    print(f"{'system':56} {'all':>12} {'harvested':>12} {'handwritten':>12} {'hw, no twin':>12} {'invalid':>7}")
    for name, s in RESULTS["test"].items():
        cell = lambda k: f"{s[k]['accuracy']:.3f} ({s[k]['correct']}/{s[k]['n']})" if k in s else "-"
        print(f"{name:56} {cell('all'):>12} {cell('harvested'):>12} {cell('handwritten'):>12} "
              f"{cell('handwritten_no_pattern_twin'):>12} {s['all']['invalid']:>7}")


def run_eval(m, tag):
    t0 = time.time()
    for key, label in (("prompt_eval", "eval prompt"), ("prompt_triage", "triage prompt")):
        preds, raws = predict(m, TEST, key)
        PRED[f"{tag}, {label}"], RAW[f"{tag}, {label}"] = preds, raws
        RESULTS["test"][f"{tag}, {label}"] = summarize(TEST, preds)
    preds, raws = predict(m, VAL_EVAL, "prompt")
    RESULTS["val"][tag] = summarize(VAL_EVAL, preds, subsets=("all",))["all"]
    RAW[f"{tag}, val"] = raws
    TIMES[f"eval_{tag.split()[0]}_s"] = round(time.time() - t0, 1)


## 4. Evaluate the base model (before fine-tuning)

`eval prompt` is the task string the PoC uses for its gold-set evaluation (the gpt-oss-120b numbers were measured with it); `triage prompt` is the one the PoC uses during triage.

In [ ]:
run_eval(model, "base " + MODEL_ID.split("/")[-1])
table()
print("validation (synthetic, in-distribution) accuracy:", {k: v["accuracy"] for k, v in RESULTS["val"].items()})


## 5. LoRA fine-tuning (fp16, loss on the answer tokens only)

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer


def as_prompt_completion(r):
    return {"prompt": messages(r["prompt"]), "completion": [{"role": "assistant", "content": r["target"]}]}


TRAIN = train_rows if TRAIN_LIMIT is None else train_rows[:TRAIN_LIMIT]
VAL = val_rows if TRAIN_LIMIT is None else val_rows[:2]
train_ds = Dataset.from_list([as_prompt_completion(r) for r in TRAIN])
val_ds = Dataset.from_list([as_prompt_completion(r) for r in VAL])
steps = MAX_STEPS if MAX_STEPS > 0 else math.ceil(len(TRAIN) / (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
per = "steps" if SMOKE else "epoch"
args = SFTConfig(
    output_dir=os.path.join(OUT_DIR, "checkpoints"), num_train_epochs=EPOCHS, max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LEARNING_RATE, lr_scheduler_type="cosine",
    warmup_steps=max(1, round(0.05 * steps)), weight_decay=0.0, max_grad_norm=1.0, optim="adamw_torch",
    fp16=DEVICE == "cuda", bf16=False, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False}, max_length=MAX_LEN, packing=False,
    completion_only_loss=True, eval_strategy=per, save_strategy=per, **({"eval_steps": MAX_STEPS,
    "save_steps": MAX_STEPS} if SMOKE else {}), load_best_model_at_end=True, metric_for_best_model="eval_loss",
    greater_is_better=False, save_total_limit=1, save_only_model=True, logging_steps=1 if SMOKE else 5,
    seed=SEED, data_seed=SEED, report_to="none")
peft_config = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, target_modules=LORA_TARGETS,
                         bias="none", task_type="CAUSAL_LM")
tok.padding_side = "right"
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, processing_class=tok,
                     peft_config=peft_config)
trainer.model.print_trainable_parameters()
t0 = time.time()
train_out = trainer.train()
TIMES["train_s"] = round(time.time() - t0, 1)
LOG = [x for x in trainer.state.log_history]
final_eval = trainer.evaluate()
assert math.isfinite(train_out.training_loss), "training loss is not finite (fp16 overflow?): see README"
print("train loss", round(train_out.training_loss, 4), "| eval loss (best checkpoint)", round(final_eval["eval_loss"], 4),
      "| best", trainer.state.best_model_checkpoint, "| minutes", round(TIMES["train_s"] / 60, 1))


## 6. Save the adapter, merge it, evaluate the fine-tuned model

In [ ]:
ADAPTER_DIR = os.path.join(OUT_DIR, "adapter")
MERGED_DIR = os.path.join(OUT_DIR, "merged")
trainer.model.save_pretrained(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)
t0 = time.time()
ft = trainer.model.merge_and_unload()
ft.config.use_cache = True
ft.save_pretrained(MERGED_DIR)
tok.padding_side = "left"
tok.save_pretrained(MERGED_DIR)
TIMES["merge_save_s"] = round(time.time() - t0, 1)
run_eval(ft, "fine-tuned " + MODEL_ID.split("/")[-1])
table()
print("validation (synthetic, in-distribution) accuracy:", {k: v["accuracy"] for k, v in RESULTS["val"].items()})


## 7. Results

In [ ]:
TIMES["total_s"] = round(time.time() - T_START, 1)
items = []
for i, r in enumerate(TEST):
    items.append({"id": r["id"], "source": r["source"], "label": r["label"], "twins_in_train": TWINS[r["id"]],
                  "pred": {name: preds[i] for name, preds in PRED.items()},
                  "raw": {name: raws[i] for name, raws in RAW.items() if not name.endswith(", val")}})
results = {
    "what": "LoRA fine-tune of an open model for regression triage; PoC gold set as test set",
    "model": MODEL_ID, "model_revision": MODEL_REVISION, "smoke_run": SMOKE, "env": ENV,
    "config": {"seed": SEED, "epochs": EPOCHS, "max_steps": MAX_STEPS, "learning_rate": LEARNING_RATE,
               "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
               "lora_dropout": LORA_DROPOUT, "lora_targets": LORA_TARGETS, "max_len": MAX_LEN,
               "max_new_tokens": MAX_NEW_TOKENS, "decoding": "greedy, repetition_penalty 1.0",
               "train_examples": len(TRAIN), "val_examples": len(VAL), "test_items": len(TEST)},
    "data": {"sha256": DATA_SHA256, "tokens": TOKENS, "splits": DATASET_REPORT["splits"],
             "leakage": DATASET_REPORT["leakage"]},
    "training": {"train_loss": train_out.training_loss, "final_eval": final_eval,
                 "best_checkpoint": trainer.state.best_model_checkpoint, "log_history": LOG},
    "test": RESULTS["test"], "val_synthetic": RESULTS["val"], "seconds": TIMES, "items": items,
    "notes": ["test = PoC gold set (10 harvested, 12 handwritten); one item is 4.5 points of 22, 8.3 of 12",
              "handwritten_no_pattern_twin = handwritten items whose decision-relevant shape (see "
              "dataset_report twin_definitions) does not occur in the training data",
              "single seed, greedy decoding; reference rows for gpt-oss-120b are copied from the PoC's own runs",
              "'invalid' = output that does not validate as TriageDecision (counted wrong); "
              "accuracy_with_rules_fallback mirrors the PoC, which uses the rules when the LLM output is invalid"],
}
RESULTS_PATH = os.path.join(OUT_DIR, "results.json")
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=1)
table()
print()
print(f"{'item':26} {'source':11} {'twins':>5} {'label':14} {'rules':14} {'base':14} {'fine-tuned':14}")
b = [k for k in PRED if k.startswith("base") and k.endswith("eval prompt")][0]
t = [k for k in PRED if k.startswith("fine-tuned") and k.endswith("eval prompt")][0]
for it in items:
    p = it["pred"]
    print(f"{it['id'][-24:]:26} {it['source']:11} {it['twins_in_train']['pattern_twins']:>5} {it['label']:14} "
          f"{p['rules (vagent.triage.classify_rules)']:14} "
          f"{p[b]:14} {p[t]:14}")
print("saved", RESULTS_PATH, "| total minutes", round(TIMES["total_s"] / 60, 1))


In [ ]:
import shutil

ADAPTER_ZIP = shutil.make_archive(os.path.join(OUT_DIR, "triage_lora_adapter"), "zip", ADAPTER_DIR)
print("download:", RESULTS_PATH, ADAPTER_ZIP, "| merged model (fp16, large):", MERGED_DIR)
try:
    from google.colab import files
    files.download(RESULTS_PATH)
    files.download(ADAPTER_ZIP)
except ImportError:
    print("On Kaggle: open the Output panel on the right (/kaggle/working/triage_lora) and download "
          "results.json and triage_lora_adapter.zip")


## 8. Optional: GGUF for local CPU use (llama.cpp)

In [ ]:
# Optional: convert the merged model to GGUF (q8_0) for local CPU inference with llama.cpp. Set RUN_GGUF = True.
# q8_0 is close to fp16 quality but it is not the model evaluated above; re-run the PoC eval on it before quoting it.
# convert_hf_to_gguf.py uses the gguf-py package from the same checkout; for Qwen it also imports sentencepiece.
if RUN_GGUF:
    import tempfile
    llama_dir = os.path.join(tempfile.gettempdir(), "llama.cpp-" + LLAMA_CPP_TAG)
    if not os.path.isdir(llama_dir):
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", LLAMA_CPP_TAG,
                               "https://github.com/ggml-org/llama.cpp", llama_dir])
    if installed("sentencepiece") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentencepiece==0.2.3"])
    GGUF_PATH = os.path.join(OUT_DIR, "triage-" + MODEL_ID.split("/")[-1].lower() + "-q8_0.gguf")
    t0 = time.time()
    subprocess.check_call([sys.executable, os.path.join(llama_dir, "convert_hf_to_gguf.py"), MERGED_DIR,
                           "--outfile", GGUF_PATH, "--outtype", "q8_0"])
    print(GGUF_PATH, round(os.path.getsize(GGUF_PATH) / 1e9, 2), "GB in", round(time.time() - t0), "s")
    try:
        from google.colab import files
        files.download(GGUF_PATH)
    except ImportError:
        pass
else:
    print("GGUF conversion skipped (set RUN_GGUF = True in the first code cell to enable)")
